# Data Workflow

### ***Nina Haide***


This project explores the AI job market in the DACH area (Germany, Austria and Switzerland) through live data collected from the Adzuna jobs API. The dataset covers a broad spectrum of AI roles - from engineering and data science to consulting, product management and system integration - reflecting the diverse career paths available to candidates with both technical and business backgrounds. The goal is to identify patterns in role demand, seniority distribution and salary transparency across different AI job categories.

442 job postings scraped live from the Adzuna API in May 2025, covering AI-related roles across Germany (361), Austria (37) and Switzerland (44). The dataset spans five role categories - AI Engineering, Data Analytics, AI Product/Project Management, AI Integration and AI Consulting - and includes job title, company, location, country, contract type, salary where disclosed, posting date and a short description excerpt. Salary data is intentionally sparse, with only 21 of 442 postings disclosing compensation figures, reflecting a well-documented pattern of salary opacity in job markets.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("All good ✅")

All good ✅


In [12]:
df = pd.read_csv("adzuna_ai_jobs_dach.csv")
df.head()

,id,title,company,location,country,category,contract_type,contract_time,salary_min,salary_max,date_posted,description,role_category,search_query,url
0,de_5708353171,Senior Engineer Machine Learning / AI (w/m/d),HUK-COBURG Versicherungsgruppe,"Coburg, Coburg (Kreis)",Germany,IT-Stellen,NaN,NaN,NaN,NaN,2026-04-23,"Hallo, lass uns WIR sagen! Wir - die HUK-COBUR...",AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/land/ad/5708353171?se=WG...
1,de_5680334179,Digital Process Engineer – AI & Automation (KI...,GPE MedTech Systeme GmbH,"Dassow, Nordwestmecklenburg (Kreis)",Germany,Beraterstellen,NaN,NaN,NaN,NaN,2026-03-27,Digital Process Engineer – AI & Automation (KI...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5680334179?utm_m...
2,de_5418313407,Senior AI / Machine Learning Engineer – Enterp...,Nortal,Deutschland,Germany,IT-Stellen,permanent,full_time,NaN,NaN,2025-09-27,Wir bei Nortal unterstützen den öffentlichen S...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5418313407?utm_m...
3,de_5698582069,AI & Machine Learning Engineer – Sensor Fusion...,Collective.work,"Altstadt-Lehel, München",Germany,Sonstige/Allgemeine Stellen,permanent,full_time,65000.0,105000.0,2026-04-14,Job description Salary: 65-105k€ (Annuel) AI &...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5698582069?utm_m...
4,de_5104382632,Staff Machine Learning Engineer (AI Platform),Smartly,"Berlin, Deutschland",Germany,IT-Stellen,NaN,full_time,NaN,NaN,2025-03-21,We are looking for a Staff Machine Learning En...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5104382632?utm_m...


In [13]:
df.count()


id               442
title            442
company          441
location         442
country          442
category         442
contract_type     65
contract_time    191
salary_min        21
salary_max        21
date_posted      442
description      442
role_category    442
search_query     442
url              442
dtype: int64

In [14]:
import re

def clean_job_titles(df):
    """Strip gender suffixes, em-dashes, ref numbers and extra whitespace from job titles."""
    df = df.copy()
    
    title = df["title"].astype(str)
    
    # Remove gender notation variants: (w/m/d), (m/w/d), (f/m/d), (KI/m/w/d), etc.
    title = title.str.replace(r"\([fmwdkiKI/]+\)", "", regex=True)
    
    # Remove em-dashes and surrounding whitespace
    title = title.str.replace(r"\s*[–—-]\s*", " ", regex=True)
    
    # Remove trailing reference numbers / job IDs (e.g. "Ref. 12345", "#42", "(m42)")
    title = title.str.replace(r"\s*(Ref\.?\s*\d+|#\d+|\(\w*\d+\w*\))", "", regex=True)
    
    # Collapse multiple spaces and strip
    title = title.str.replace(r"\s{2,}", " ", regex=True).str.strip()
    
    df["title_clean"] = title
    return df

df = clean_job_titles(df)
df[["title", "title_clean"]].head(10)

,title,title_clean
0,Senior Engineer Machine Learning / AI (w/m/d),Senior Engineer Machine Learning / AI
1,Digital Process Engineer – AI & Automation (KI...,Digital Process Engineer AI & Automation (KI E...
2,Senior AI / Machine Learning Engineer – Enterp...,Senior AI / Machine Learning Engineer Enterpri...
3,AI & Machine Learning Engineer – Sensor Fusion...,AI & Machine Learning Engineer Sensor Fusion &...
4,Staff Machine Learning Engineer (AI Platform),Staff Machine Learning Engineer (AI Platform)
5,Senior Machine Learning Engineer (AI Platform),Senior Machine Learning Engineer (AI Platform)
6,Senior Engineer Machine Learning / AI (w/m/d),Senior Engineer Machine Learning / AI
7,Senior Engineer Machine Learning / AI (w/m/d),Senior Engineer Machine Learning / AI
8,Senior Machine Learning/AI Engineer (m/w/d),Senior Machine Learning/AI Engineer
9,Machine Learning Engineer - AI Insights (gn) f...,Machine Learning Engineer AI Insights (gn) ful...


In [15]:
HOURLY_THRESHOLD = 500  # values below this are almost certainly hourly rates, not annual

def clean_salary_data(df):
    """Flag implausible salaries, add availability flag, compute midpoint."""
    df = df.copy()
    
    # Flag rows where salary looks like an hourly rate (e.g. 80.0)
    df["salary_likely_hourly"] = (
        df["salary_min"].notna() & (df["salary_min"] < HOURLY_THRESHOLD)
    ) | (
        df["salary_max"].notna() & (df["salary_max"] < HOURLY_THRESHOLD)
    )
    
    # Null out implausible values so they don't skew analysis
    df.loc[df["salary_likely_hourly"], ["salary_min", "salary_max"]] = None
    
    # Boolean flag: does this row have any usable salary data?
    df["salary_available"] = df["salary_min"].notna() | df["salary_max"].notna()
    
    # Midpoint: average of min+max where both exist; fall back to whichever is present
    df["salary_midpoint"] = df[["salary_min", "salary_max"]].mean(axis=1)
    
    return df

df = clean_salary_data(df)

print(f"Salary available:    {df['salary_available'].sum()} / {len(df)} rows")
print(f"Likely hourly rate:  {df['salary_likely_hourly'].sum()} rows nulled out")
print(f"Midpoint range:      €{df['salary_midpoint'].min():,.0f} – €{df['salary_midpoint'].max():,.0f}")
df[["title_clean", "salary_min", "salary_max", "salary_midpoint", "salary_available"]].dropna(subset=["salary_midpoint"]).head()

Salary available:    18 / 442 rows
Likely hourly rate:  3 rows nulled out
Midpoint range:      €10,818 – €416,000


,title_clean,salary_min,salary_max,salary_midpoint,salary_available
3,AI & Machine Learning Engineer Sensor Fusion &...,65000.0,105000.0,85000.0,True
11,Full Stack Engineer,75000.0,100000.0,87500.0,True
19,Staff Backend Developer (Python),80000.0,110000.0,95000.0,True
24,Sr. Product Manager,115000.0,130000.0,122500.0,True
25,Staff Product Manager,115000.0,130000.0,122500.0,True
